# Лаборатория 5. Измеряем качество числом

**Что мы сделаем:** возьмём бота из темы 2, составим для него эталонный набор вопросов
и посчитаем, сколько ответов правильные. Потом попробуем улучшение — и проверим числом,
помогло оно или навредило.

Потом разберём провалы: кто виноват в каждом — поиск или модель, и почему. А в конце
поручим проверку самой модели («ИИ-судья») и посмотрим, можно ли ей доверять.

**Что понадобится:** код класса от учителя.

In [ ]:
!pip -q install rank_bm25 openai pandas

In [ ]:
import getpass
import os
from pprint import pprint

import pandas as pd
from openai import OpenAI
from rank_bm25 import BM25Okapi

ADRES = "https://ai9.adelfos.ru/api/v1"
MODEL = "qwen/qwen3.7-flash"

try:
    from google.colab import userdata
    KOD_KLASSA = userdata.get("AI9_KOD") or os.environ.get("AI9_KOD")
except Exception:
    KOD_KLASSA = os.environ.get("AI9_KOD")

# Сервер проверит код, только когда мы обратимся к нему с ключом. Поэтому делаем один
# лёгкий запрос (список моделей) и, если код не принят, спрашиваем его заново.
client = None
while client is None:
    if not KOD_KLASSA:
        KOD_KLASSA = getpass.getpass("Код класса: ")
    client = OpenAI(base_url=ADRES, api_key=KOD_KLASSA)
    try:
        client.models.list()   # неверный код сервер не примет и ответит ошибкой
        print("Всё хорошо: код подошёл. Модель:", MODEL)
    except Exception:
        print("Код не подошёл — проверь его у учителя и введи заново.")
        client = None
        KOD_KLASSA = None      # после ошибки код из секретов и окружения больше не берём

## Шаг 1. Бот из темы 2

Это тот же самый бот: чанки, поиск BM25, ответ модели с запретом выдумывать.
Ничего нового, просто собрали в одном месте, чтобы было что измерять.

In [ ]:
PRAVILA = """
Кружок робототехники работает по вторникам и четвергам. Начало в 15:40, кабинет 204.
С собой нужна тетрадь, остальное выдают на месте.

Кружок рисования собирается по средам в 16:00 в кабинете 112. Краски и кисти каждый
приносит свои, бумагу выдаёт школа.

Столовая работает с 9:00 до 15:00. Завтрак с 9:00 до 9:30, обед с 12:20 до 13:00.

Библиотека открыта с 8:30 до 17:00 каждый день, кроме пятницы. Книги выдают на две недели.

Школьный спортзал открыт до 19:00. Секции: волейбол по понедельникам, баскетбол по четвергам.
"""

CHANKI = [k.strip().replace("\n", " ") for k in PRAVILA.strip().split("\n\n")]


def v_slova(text):
    return "".join(b.lower() if b.isalnum() else " " for b in text).split()


poisk_obychnyy = BM25Okapi([v_slova(c) for c in CHANKI])
print(f"Чанков: {len(CHANKI)}")

## Шаг 2. Эталонный набор

> **Эталонный набор** — список вопросов с правильными ответами, составленный заранее
> и не меняющийся, на котором проверяют систему после каждого изменения.

Три правила, по которым он составлен:

1. Вопросы **разные по трудности**: есть простые, есть «другими словами».
2. Обязательно есть вопросы, **ответа на которые нет** в документах, — иначе не заметишь,
   что бот начал выдумывать.
3. Для каждого вопроса известно, **какой чанк** должен найтись. Это позволяет измерять
   поиск и ответ по отдельности: если ответ неверный, сразу видно, чья вина.

In [ ]:
# nuzhen_chank = None означает: правильное поведение — не найти ничего и сказать «нет данных».
ETALON = [
    {"vopros": "Во сколько начинается кружок робототехники?", "nuzhen_chank": 0, "otvet_soderzhit": "15:40"},
    {"vopros": "В каком кабинете рисование?",                 "nuzhen_chank": 1, "otvet_soderzhit": "112"},
    {"vopros": "Что нужно приносить на рисование?",           "nuzhen_chank": 1, "otvet_soderzhit": "краск"},
    {"vopros": "До скольких работает библиотека?",            "nuzhen_chank": 3, "otvet_soderzhit": "17:00"},
    {"vopros": "Когда можно позавтракать в школе?",           "nuzhen_chank": 2, "otvet_soderzhit": "9:00"},
    {"vopros": "В какой день баскетбол?",                     "nuzhen_chank": 4, "otvet_soderzhit": "четверг"},
    {"vopros": "На сколько дней выдают книги?",               "nuzhen_chank": 3, "otvet_soderzhit": "две недел"},
    {"vopros": "Сколько стоит проезд в школьном автобусе?",   "nuzhen_chank": None, "otvet_soderzhit": None},
    {"vopros": "Когда родительское собрание?",                "nuzhen_chank": None, "otvet_soderzhit": None},
]

print(f"В эталонном наборе {len(ETALON)} вопросов, "
      f"из них {sum(1 for e in ETALON if e['nuzhen_chank'] is None)} — без ответа в документах.")

## Шаг 3. Измеряем ПОИСК отдельно

Первый и самый дешёвый замер: находится ли нужный чанк вообще. Модель тут не нужна —
значит, замер бесплатный и мгновенный, его можно гонять хоть после каждой правки.

Считаем долю вопросов, где всё прошло правильно: нужный чанк попал в найденное,
а для вопросов без ответа — не нашлось ничего.

In [ ]:
def izmerit_poisk(poisk_obj, chanki, skolko=2, nazvanie=""):
    verno, provaly = 0, []
    for zapis in ETALON:
        ocenki = poisk_obj.get_scores(v_slova(zapis["vopros"]))
        poryadok = sorted(range(len(chanki)), key=lambda i: ocenki[i], reverse=True)
        naydeno = [i for i in poryadok[:skolko] if ocenki[i] > 0]

        if zapis["nuzhen_chank"] is None:
            uspekh = len(naydeno) == 0
        else:
            uspekh = zapis["nuzhen_chank"] in naydeno

        verno += uspekh
        if not uspekh:
            provaly.append((zapis["vopros"], zapis["nuzhen_chank"], naydeno))

    dolya = verno / len(ETALON)
    print(f"{nazvanie}: {verno} из {len(ETALON)} = {dolya:.0%}")
    for vopros, nuzhen, naydeno in provaly:
        print(f"   ✗ {vopros!r}: нужен чанк {nuzhen}, найдено {naydeno}")
    return dolya


kachestvo_do = izmerit_poisk(poisk_obychnyy, CHANKI, nazvanie="Поиск как есть")

Обрати внимание на список провалов — он важнее самого числа. Само по себе «67 %»
ничего не говорит; а вот «провалились ровно те вопросы, где слово стоит в другой
форме» — это уже понятная причина, которую можно чинить.

## Шаг 4. Пробуем улучшение и проверяем его числом

Гипотеза из темы 2: поиск спотыкается на словоформах («рисование» против «рисования»).
Попробуем обрезать слова до основы — грубо, первые 6 букв.

Это и есть главный рабочий приём инженера: **изменил одно — замерил на том же наборе.**

In [ ]:
def v_slova_obrezannye(text):
    return [s[:6] for s in v_slova(text)]


# Важно: и документы, и вопрос должны обрабатываться ОДИНАКОВО, иначе сравнивать нечего.
poisk_obrezannyy = BM25Okapi([v_slova_obrezannye(c) for c in CHANKI])


def izmerit_poisk_obrezannyy():
    verno, provaly = 0, []
    for zapis in ETALON:
        ocenki = poisk_obrezannyy.get_scores(v_slova_obrezannye(zapis["vopros"]))
        poryadok = sorted(range(len(CHANKI)), key=lambda i: ocenki[i], reverse=True)
        naydeno = [i for i in poryadok[:2] if ocenki[i] > 0]
        uspekh = (len(naydeno) == 0) if zapis["nuzhen_chank"] is None else (zapis["nuzhen_chank"] in naydeno)
        verno += uspekh
        if not uspekh:
            provaly.append((zapis["vopros"], zapis["nuzhen_chank"], naydeno))
    print(f"Поиск с обрезкой слов: {verno} из {len(ETALON)} = {verno/len(ETALON):.0%}")
    for vopros, nuzhen, naydeno in provaly:
        print(f"   ✗ {vopros!r}: нужен чанк {nuzhen}, найдено {naydeno}")
    return verno / len(ETALON)


kachestvo_posle = izmerit_poisk_obrezannyy()

print()
if kachestvo_posle > kachestvo_do:
    print(f"✅ Улучшение помогло: {kachestvo_do:.0%} → {kachestvo_posle:.0%}. Оставляем.")
elif kachestvo_posle == kachestvo_do:
    print(f"➖ Ничего не изменилось ({kachestvo_do:.0%}). Усложнять без пользы не нужно.")
else:
    print(f"❌ Стало хуже: {kachestvo_do:.0%} → {kachestvo_posle:.0%}. Откатываем.")

Вот ради чего всё затевалось. Без замера ты бы сказал «вроде стало лучше» — и остался
бы с этим ощущением. С замером у тебя есть число и список конкретных провалов.

Заметь ещё одну вещь: улучшение могло **испортить** вопросы без ответа. Обрезка слов
делает поиск «щедрее», и он начинает находить что-нибудь там, где находить нечего.
Это типичный размен: чинишь одно — ломаешь другое. Увидеть его можно только на наборе.

## Шаг 5. Измеряем ОТВЕТ

Поиск — половина дела. Теперь проверим, что модель отвечает по найденному правильно.

Простейшая проверка — обычным кодом: содержит ли ответ нужный факт. Она бесплатная,
мгновенная и, в отличие от ИИ-судьи, не имеет своего мнения.

In [ ]:
PRAVILO = (
    "Ты помощник школы. Отвечай ТОЛЬКО по тексту из блока ДОКУМЕНТЫ.\n"
    "Если ответа в документах нет — ответь ровно: «В документах этого нет».\n"
    "Отвечай одним предложением."
)


def otvetit(vopros, pokazyvat_zapros=False):
    ocenki = poisk_obrezannyy.get_scores(v_slova_obrezannye(vopros))
    poryadok = sorted(range(len(CHANKI)), key=lambda i: ocenki[i], reverse=True)
    naydeno = [CHANKI[i] for i in poryadok[:2] if ocenki[i] > 0]
    if not naydeno:
        return "В документах этого нет."

    dokumenty = "\n".join(f"- {c}" for c in naydeno)
    zapros = [{"role": "system", "content": PRAVILO},
              {"role": "user", "content": f"ДОКУМЕНТЫ:\n{dokumenty}\n\nВОПРОС: {vopros}"}]

    # Полезно хотя бы раз посмотреть, что именно уходит модели: ответ бывает
    # неправильным не потому, что модель плоха, а потому что ей прислали не то.
    if pokazyvat_zapros:
        print("Что уходит модели:")
        pprint(zapros, width=100, sort_dicts=False)
        print()

    otvet = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=150, messages=zapros)
    return otvet.choices[0].message.content.strip()


# Посмотрим на один запрос целиком — дальше будем печатать только ответы.
print("Пример полного запроса:\n")
primer = otvetit(ETALON[0]["vopros"], pokazyvat_zapros=True)
print("Ответ модели:", primer)


stroki = []
for zapis in ETALON:
    otvet = otvetit(zapis["vopros"])
    zhdem = zapis["otvet_soderzhit"]

    if zhdem is None:
        # Ждём отказ. Проверяем по ключевым словам отказа.
        verno = "нет" in otvet.lower() and "документ" in otvet.lower()
    else:
        verno = zhdem.lower() in otvet.lower()

    stroki.append({"вопрос": zapis["vopros"][:38], "ответ бота": otvet[:44],
                   "ждём": zhdem or "отказ", "верно": "✅" if verno else "❌",
                   "_polnyy_otvet": otvet})

tablica = pd.DataFrame(stroki).drop(columns=["_polnyy_otvet"])
print(f"Правильных ответов: {sum(s['верно'] == '✅' for s in stroki)} из {len(stroki)}")
tablica

Такая проверка называется **проверкой по ключу**: мы заранее знаем, какое слово или
число обязано быть в правильном ответе, и просто ищем его в тексте.

Она грубая: ответ «кружок начинается не в 15:40, а раньше» тоже пройдёт проверку,
хотя он неверный. Зато она честная, бесплатная и работает мгновенно.

**Правило:** сначала проверяй кодом всё, что можно проверить кодом. И только то,
что коду не по силам, отдавай следующему способу.

## Шаг 6. Разбор провалов: кто виноват и почему

Число «7 из 9» говорит, **что** что-то не так, но не говорит **что именно**. Возьмём
каждый ❌ и ответим на два вопроса:

1. **Кто виноват?** Если поиск не принёс нужный чанк, модель ни при чём: ей просто
   нечем было ответить. Если чанк был, а ответ неверный, — виновата модель или правило.
2. **Почему?** Для провала поиска посмотрим «под лупой», какие слова совпали с каждым чанком.

In [ ]:
def nayti_chanki(vopros, v_slova_func, poisk_obj, skolko=2):
    ocenki = poisk_obj.get_scores(v_slova_func(vopros))
    poryadok = sorted(range(len(CHANKI)), key=lambda i: ocenki[i], reverse=True)
    return [i for i in poryadok[:skolko] if ocenki[i] > 0]


print(f"{'вопрос':<44} {'нужен':>5} {'найдено':>8}  {'ответ':<6} виноват")
for zapis, stroka in zip(ETALON, stroki):
    naydeno = nayti_chanki(zapis["vopros"], v_slova_obrezannye, poisk_obrezannyy)
    nuzhen = zapis["nuzhen_chank"]
    poisk_ok = (not naydeno) if nuzhen is None else (nuzhen in naydeno)
    otvet_ok = stroka["верно"] == "✅"

    if poisk_ok and otvet_ok:
        vinovnik = "—"
    elif not poisk_ok and nuzhen is not None:
        vinovnik = "ПОИСК: нужный чанк не нашёлся"
    elif not poisk_ok:
        vinovnik = "поиск нашёл лишнее" + ("" if otvet_ok else " → и модель поверила")
    else:
        vinovnik = "МОДЕЛЬ: чанк был, ответ неверный"
    print(f"{zapis['vopros'][:44]:<44} {str(nuzhen):>5} {str(naydeno):>8}  {stroka['верно']:<6} {vinovnik}")

**Как читать таблицу:**

* **«ПОИСК: нужный чанк не нашёлся»** — самый частый вид провала. Модель, скорее всего,
  ответила честно «в документах этого нет»: ей прислали не те документы. Чинить нужно
  **поиск**, а правило для модели трогать бесполезно.
* **«поиск нашёл лишнее»** у вопроса без ответа — поиск принёс чанк, где ответа нет.
  Если модель всё равно сказала «нет» — ответ засчитан, но это везение, а не заслуга
  поиска. Если поверила и выдумала — провал уже двойной.
* **«МОДЕЛЬ: чанк был, ответ неверный»** — вот здесь чинят правило, запрос или модель.

Возьмём самый поучительный провал — вопрос про завтрак — и посмотрим под лупой.

In [ ]:
def pod_lupoy(vopros, v_slova_func, poisk_obj, nazvanie):
    slova_voprosa = v_slova_func(vopros)
    ocenki = poisk_obj.get_scores(slova_voprosa)
    print(f"--- {nazvanie} ---")
    print(f"вопрос превратился в слова: {slova_voprosa}")
    for i, chank in enumerate(CHANKI):
        obshchie = sorted(set(slova_voprosa) & set(v_slova_func(chank)))
        metka = "  ← нужный" if "Завтрак" in chank else ""
        print(f"   чанк {i}: оценка {ocenki[i]:.3f}, общие слова {obshchie or '—'}  | {chank[:40]}…{metka}")
    print()


VOPROS_ZAVTRAK = "Когда можно позавтракать в школе?"
pod_lupoy(VOPROS_ZAVTRAK, v_slova_obrezannye, poisk_obrezannyy, "поиск с обрезкой слов (как в боте)")
print("А вот слова нужного чанка:", v_slova_obrezannye(CHANKI[2]))

**Что случилось — по шагам:**

1. В нужном чанке написано «**Завтрак**», после обрезки — `завтра`. В вопросе
   «**позавтракать**» → `позавт`. Для компьютера это разные слова: мешает приставка «по».
   Общих слов с нужным чанком — ноль, оценка 0.
2. Зато у чанков про кружки нашлось общее слово — предлог **«в»**. Совпадение на предлоге
   ничего не значит, но поиск этого не знает и приносит эти чанки.
3. Модель получила документы про кружки, завтрака там нет — и **правильно** ответила
   «в документах этого нет». Проверка по ключу ждала «9:00» и поставила ❌.

Итог: модель не виновата, поиск спотыкается дважды — на приставке и на предлоге.
Попробуем починить оба места и **замерим** каждое исправление на всём эталонном наборе.

### Исправление А: выкинуть служебные слова

Предлоги, союзы и частицы встречаются почти везде и смысла для поиска не несут.
Их принято выбрасывать до поиска.

> **Стоп-слова** — частые служебные слова (предлоги, союзы, частицы), которые убирают
> из текста перед поиском, потому что совпадение на них ничего не говорит о смысле.

In [ ]:
STOP_SLOVA = {"в", "во", "на", "с", "со", "по", "до", "за", "и", "а", "но", "к", "у", "о",
              "об", "от", "из", "для", "не", "ли", "же", "бы"}


def v_slova_bez_stop(text):
    return [s[:6] for s in v_slova(text) if s not in STOP_SLOVA]


poisk_bez_stop = BM25Okapi([v_slova_bez_stop(c) for c in CHANKI])
pod_lupoy(VOPROS_ZAVTRAK, v_slova_bez_stop, poisk_bez_stop, "без стоп-слов")

Предлог больше не совпадает — чужие чанки про кружки не нашлись. Нужный чанк **тоже**
не нашёлся: приставка никуда не делась. Но это уже честнее: поиск не принёс ничего,
и бот скажет «в документах этого нет», **даже не вызывая модель**. Тот же ответ,
но бесплатно.

### Исправление Б: отрезать приставки

Раз мешает «по-», попробуем грубое правило: если слово начинается с частой приставки
и после неё остаётся хотя бы 5 букв — отрезать приставку.

In [ ]:
PRISTAVKI = ("по", "за", "на", "вы", "пере", "при", "от", "до")


def v_slova_bez_pristavok(text):
    rezultat = []
    for slovo in v_slova(text):
        if slovo in STOP_SLOVA:
            continue
        for p in PRISTAVKI:
            if slovo.startswith(p) and len(slovo) - len(p) >= 5:
                slovo = slovo[len(p):]
                break
        rezultat.append(slovo[:6])
    return rezultat


poisk_bez_pristavok = BM25Okapi([v_slova_bez_pristavok(c) for c in CHANKI])
pod_lupoy(VOPROS_ZAVTRAK, v_slova_bez_pristavok, poisk_bez_pristavok, "без стоп-слов и приставок")
print("Слова нужного чанка теперь:", v_slova_bez_pristavok(CHANKI[2]))

Посмотри внимательно на последнюю строку. Из вопроса «позавтракать» правило сделало
`завтра` — ровно то, что нужно. Но **то же правило** испортило сам чанк: «**за**втрак»
начинается с «за», и получилось `втрак`. Совпадения снова нет.

Это классическая ловушка грубых правил: «за» в «завтраке» — не приставка, но правило
этого не знает. Чиним одно слово — ломаем другое.

### Замер всех вариантов на всём наборе

Одно слово — не доказательство. Проверим каждый вариант на всём эталонном наборе,
как положено (шаг 4).

In [ ]:
def zamer_poiska(v_slova_func, nazvanie):
    poisk_obj = BM25Okapi([v_slova_func(c) for c in CHANKI])
    verno, provaly = 0, []
    for zapis in ETALON:
        naydeno = nayti_chanki(zapis["vopros"], v_slova_func, poisk_obj)
        ok = (not naydeno) if zapis["nuzhen_chank"] is None else (zapis["nuzhen_chank"] in naydeno)
        verno += ok
        if not ok:
            provaly.append(f"{zapis['vopros'][:40]} (нужен {zapis['nuzhen_chank']}, найдено {naydeno})")
    print(f"{nazvanie:<28} {verno} из {len(ETALON)}")
    for p in provaly:
        print(f"{'':<28}   ✗ {p}")


zamer_poiska(v_slova, "как есть")
zamer_poiska(v_slova_obrezannye, "обрезка слов")
zamer_poiska(v_slova_bez_stop, "+ без стоп-слов")
zamer_poiska(v_slova_bez_pristavok, "+ без приставок")

**Что посмотреть в выводе:**

1. Число «из 9» у последних вариантов почти не меняется — а провалы **разные**. У обрезки
   вопрос про завтрак приносит чужие чанки, без стоп-слов — не приносит ничего. Для
   ученика разница огромная: «бот нашёл что-то не то» против «бот честно не знает».
   Одно число это скрывает — поэтому смотрят и на список провалов.
2. Посмотри на вопрос про **школьный автобус** — у него нет ответа, но поиск находит
   чанк про спортзал: общее слово `школьн` («школьном» и «Школьный»). Совпадение
   по смыслу случайное, и стоп-слова тут не помогут — это не служебное слово.
3. Отрезание приставок **ничего не дало**: как ты видел, оно ломает слова, в которых
   «за» или «по» — вовсе не приставка.

**Главный вывод разбора.** «Позавтракать» и «завтрак» связаны смыслом, а не буквами.
Поиск по словам можно подкручивать бесконечно — он всё равно сравнивает буквы. Настоящее
исправление — **поиск по смыслу** из темы 3: там «когда поесть утром» находит завтрак
вообще без общих слов. А вопрос про завтрак остаётся в эталонном наборе: именно он
покажет, помогло ли исправление.

**Вариант:** замени вопрос на «Во сколько завтрак?» и запусти `pod_lupoy`. Почему
теперь нашлось?

## Шаг 7. ИИ-судья и его слабости

А что, если ответ правильный, но сформулирован иначе, чем эталон? Проверка по ключу
такой ответ забракует. Поручим сравнение самой модели.

> **ИИ-судья** — модель, которой поручили оценить ответ другой модели по заданным правилам.

Обрати внимание, как сформулирован запрос: судье не говорят «оцени качество от 1 до 10»
(слишком расплывчато, оценки будут плавать), а задают **конкретный вопрос** с ответом
ДА или НЕТ. Чем уже вопрос, тем меньше судья фантазирует.

In [ ]:
def sudya(vopros, otvet_bota, etalonnyy_fakt, pokazyvat_zapros=False):
    """Совпадает ли ответ бота с эталонным фактом по СУТИ. Отвечает ДА или НЕТ."""
    zapros = [
        {"role": "system", "content":
            "Ты строгий проверяющий. Отвечай ровно одним словом: ДА или НЕТ. "
            "ДА — если ответ содержит указанный факт. НЕТ — если не содержит или противоречит."},
        {"role": "user", "content":
            f"ВОПРОС: {vopros}\nОТВЕТ БОТА: {otvet_bota}\nФАКТ, который должен быть: {etalonnyy_fakt}"},
    ]
    if pokazyvat_zapros:
        print("Что видит судья:")
        pprint(zapros, width=100, sort_dicts=False)
        print()

    reshenie = client.chat.completions.create(
        model=MODEL, temperature=0, max_tokens=10, messages=zapros)
    return (reshenie.choices[0].message.content or "").strip().upper().startswith("ДА")


# Один раз покажем целиком, что именно получает судья.
sudya("Во сколько робототехника?", "Кружок начинается в 15:40.", "15:40", pokazyvat_zapros=True)


# Проверим судью на заведомо известных случаях — в том числе на подставных.
proby = [
    ("Во сколько робототехника?", "Кружок начинается в 15:40.", "15:40", True),
    ("Во сколько робототехника?", "Кружок начинается в 16:20.", "15:40", False),
    ("Во сколько робототехника?", "Занятия стартуют без двадцати четыре дня.", "15:40", True),
    ("Когда баскетбол?", "По четвергам в спортзале.", "четверг", True),
]

print("Проверяем самого судью (его оценки против того, что мы знаем сами):\n")
sovpalo = 0
for vopros, otvet_bota, fakt, pravilno_li in proby:
    reshenie = sudya(vopros, otvet_bota, fakt)
    sovpadenie = reshenie == pravilno_li
    sovpalo += sovpadenie
    print(f"{'✅' if sovpadenie else '❌'} судья сказал {'ДА' if reshenie else 'НЕТ'}, "
          f"а правильно {'ДА' if pravilno_li else 'НЕТ'} — {otvet_bota[:45]}")

print(f"\nСудья совпал с человеком в {sovpalo} случаях из {len(proby)}.")

Это и называется **сверкой судьи**: прежде чем доверять его оценкам, проверяют их
на примерах, где правильный ответ известен заранее. Если судья ошибается уже здесь —
доверять его оценкам на тысяче ответов тем более нельзя.

Третий пример — самый интересный. «Без двадцати четыре дня» это и есть 15:40, просто
словами. Проверка по ключу такой ответ забракует — она ищет строку «15:40». Судью
заводят именно ради таких случаев.

И очень может быть, что он **тоже ошибся** на нём. У меня при подготовке этой
лаборатории ошибся: сказал НЕТ там, где правильный ответ ДА.

Вот почему сверка не формальность. Судья — такая же модель, со всеми свойствами
из темы 1: он не проверяет факты, он продолжает текст правдоподобным образом.
Пока ты не сверил его с собой на десятке примеров, его оценки — просто числа.

Но у него есть известные слабости, которые исследователи находят снова и снова:

| Слабость | В чём проявляется |
|---|---|
| любит длинное | длинный ответ получает оценку выше короткого при том же содержании |
| любит уверенное | уверенный тон ценится выше осторожного, даже если факты хуже |
| смотрит на порядок | из двух ответов чаще выбирает первый |
| хвалит своих | ответ такой же модели оценивает выше |

Отсюда практический вывод: если бот «стал отвечать лучше», а изменилось только то,
что ответы стали длиннее и увереннее, — скорее всего, улучшился не бот, а его оценка.

## Попробуй сам

1. Добавь в `ETALON` три своих вопроса, обязательно один без ответа в документах.
   Изменилось ли качество? Почему число может упасть, хотя бот не менялся?
2. Поставь в `izmerit_poisk` `skolko=3`. Качество поиска выросло — а что стало
   с вопросами без ответа?
3. Испорти `PRAVILO`: убери строчку про «В документах этого нет» и замерь снова.
   На сколько упало качество?
4. Дай судье заведомо длинный и уверенный, но неправильный ответ. Поймает ли он подвох?
5. Добавь в `STOP_SLOVA` слово «когда» и замерь поиск снова. Что стало с вопросами
   «когда каникулы»-типа? Почему со стоп-словами тоже нужно осторожно?

## Что унести с собой

* «Вроде стало лучше» — не результат. Результат — число на **фиксированном** наборе.
* **Эталонный набор** составляют заранее и не меняют, пока идут сравнения.
* В наборе обязательно есть вопросы без ответа — иначе не поймаешь выдумки.
* Поиск и ответ измеряют **по отдельности**: иначе не поймёшь, что чинить.
* Список провалов полезнее самого числа: в нём видна причина.
* Для каждого провала сначала отвечают, **кто виноват**: поиск или модель.
* Грубые правила для слов (стоп-слова, приставки) помогают и ломают одновременно —
  каждое проверяют замером; связь по смыслу чинит только поиск по смыслу.
* Улучшая одно, часто ломаешь другое — это видно только на наборе.
* Что можно проверить кодом — проверяй кодом: быстро, бесплатно, без чужого мнения.
* **ИИ-судья** нужен там, где код бессилен, но его самого надо сверять с человеком.